In [1]:
import paho.mqtt.client as mqtt
import time
import re

BROKER = "127.0.0.1"
TOPIC = "imu/data"
TARGET = 1000   # number of packets to collect

seq_list = []
collect = True

# -------------------------------------------------------
# MQTT Callbacks
# -------------------------------------------------------
def on_connect(client, userdata, flags, rc):
    if rc == 0:
        print("Connected to MQTT broker.")
        client.subscribe(TOPIC)
    else:
        print("Failed to connect:", rc)

def on_message(client, userdata, msg):
    global seq_list, collect

    if not collect:
        return

    payload = msg.payload.decode().strip()
    # Example message format:
    # imu/data [[98271797,2241,0.20,-0.97,-0.17,0.24,-0.85,0.06]]

    # Extract full content inside [[ ... ]]
    match = re.findall(r"\[([0-9\.\,\-\s]+)\]", payload)
    if not match:
        return

    # Split into numbers
    values = [float(v) for v in match[0].split(",")]

    # Expecting 8 numbers: ts, seq, ax, ay, az, gx, gy, gz
    if len(values) != 8:
        return

    seq = int(values[1])   # <- SEQUENCE NUMBER IS NOW THE SECOND VALUE

    seq_list.append(seq)

    print(f"Received: {len(seq_list)}/{TARGET}", end="\r")


# -------------------------------------------------------
# Run Test
# -------------------------------------------------------
client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message

print("Starting packet loss test...")
client.connect(BROKER, 1883, 60)
client.loop_start()

start = time.time()

# Timeout = 40 seconds or until 1000 packets collected
while len(seq_list) < TARGET and (time.time() - start) < 40:
    time.sleep(0.01)

collect = False
client.loop_stop()
client.disconnect()

print("\n--- Collection Finished ---")

# -------------------------------------------------------
# Analysis
# -------------------------------------------------------
if len(seq_list) == 0:
    print("ERROR: No packets received.")
    exit()

unique = sorted(set(seq_list))

min_seq = unique[0]
max_seq = unique[-1]

expected = max_seq - min_seq + 1
received = len(unique)
missing = expected - received

loss_pct = (missing / expected) * 100

print("\n======= MQTT Packet Loss Report =======")
print(f"Sequence range:     {min_seq} → {max_seq}")
print(f"Expected messages:  {expected}")
print(f"Received unique:    {received}")
print(f"Missing:            {missing}")
print(f"Loss percent:       {loss_pct:.2f}%")
print("=======================================")

if loss_pct <= 5:
    print("PASS ✓ (<=5% loss)")
else:
    print("FAIL ✗ (>5% loss)")


Starting packet loss test...
Connected to MQTT broker.


/var/folders/m1/qxpcclg55hg_8gcmt4vpkrq80000gn/T/ipykernel_10740/3904594057.py:54: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


Received: 1000/1000
--- Collection Finished ---

======= MQTT Packet Loss Report =======
Sequence range:     2217 → 3216
Expected messages:  1000
Received unique:    1000
Missing:            0
Loss percent:       0.00%
PASS ✓ (<=5% loss)
